# Explore a Numerai LightGBM tree with SuperTree

Choose one of the five seeds trained in `hidden-mistakes.ipynb`, then inspect any tree in its saved LightGBM model. The repository includes a 500-row v5.3 quantum-feature sample, so this notebook needs no full Numerai download and **does not retrain**.

Run this notebook with the `v5` kernel. The interactive visualization is rendered by SuperTree.

In [ ]:
from pathlib import Path
import hashlib
import json

import lightgbm as lgb
import pandas as pd
from supertree import SuperTree

DATA = Path('data-v5.3')
ARTIFACTS = Path('hidden-mistakes-artifacts/3e0cbf61aefc')
SEED = 1             # One of: 1, 7, 23, 42, 89
TREE_INDEX = 0       # Zero-based LightGBM tree number
START_DEPTH = 2

print('Selected seed:', SEED, '| tree:', TREE_INDEX)

## Load the saved model

The signature records the training eras and ordered feature contract. We recover feature names from the saved model and check the contract before visualizing the tree.

In [ ]:
signature_path = ARTIFACTS / 'signature.json'
assert signature_path.exists(), 'Run hidden-mistakes.ipynb first, then set ARTIFACTS to its reported directory.'
signature = json.loads(signature_path.read_text())
assert SEED in signature['seeds'], f'Seed {SEED} is not in this run: {signature["seeds"]}'
target = signature['target']
model_path = ARTIFACTS / f'lightgbm_seed_{SEED}.txt'
assert model_path.exists(), f'Missing trained model: {model_path}'
model = lgb.Booster(model_file=str(model_path))
features = model.feature_name()
assert hashlib.sha256('\n'.join(features).encode()).hexdigest() == signature['feature_hash'], 'Model feature order differs from the training signature'
assert 0 <= TREE_INDEX < model.num_trees(), f'Choose TREE_INDEX from 0 to {model.num_trees() - 1}'
tree = model.dump_model()['tree_info'][TREE_INDEX]['tree_structure']
root_feature = features[tree['split_feature']] if 'split_feature' in tree else '(leaf only)'
print(f'Model: {model_path}')
print(f'Trees: {model.num_trees()} | selected tree: {TREE_INDEX} | root feature: {root_feature}')

## Read a small training sample

SuperTree uses sample rows to show how observations flow through the tree. The bundled 500-row sample comes from training eras 1, 5, and 9. It is for visualization only; no full training parquet or `features.json` is required.

In [ ]:
sample_path = DATA / 'supertree-quantum-500.parquet'
assert sample_path.exists(), f'Missing bundled demo sample: {sample_path}'
sample_frame = pd.read_parquet(sample_path)
assert list(sample_frame.columns) == ['era', *features, target], 'Sample feature order differs from the saved model'
assert sample_frame[target].notna().all(), 'Sample contains missing target values'
sample_eras = sorted(sample_frame['era'].astype(int).unique().tolist())
assert set(sample_eras).issubset(signature['train_eras']), 'Sample contains eras outside model training'
X_sample = sample_frame[features]
y_sample = sample_frame[target]
print(f'Visualization sample: {len(X_sample):,} rows from training eras {sample_eras}')
print(f'Feature columns: {len(X_sample.columns)} | target: {target}')

## Show the selected tree

Change `SEED` or `TREE_INDEX` in the first code cell and rerun to explore another tree. `TREE_INDEX=0` is the first tree used in the seed comparison of `hidden-mistakes.ipynb`.

In [ ]:
tree_view = SuperTree(
    model, X_sample, y_sample, feature_names=features, target_names=target,
)
tree_view.show_tree(
    which_tree=TREE_INDEX, start_depth=START_DEPTH, max_samples=len(X_sample),
)